# Train Rainbow DQN on Kaggle / Colab

**Kaggle:** *Settings → Accelerator: GPU T4 / P100*, *Internet: on*. Upload the project folder as a
dataset (zip the `RL` folder → *Add data → Upload*), or put it on GitHub and set `REPO` below.
**Colab:** *Runtime → Change runtime type → GPU*, then upload `RL.zip` to the file panel.

Kaggle sessions stop after ~12 h, so training uses `--time-limit` and `--resume`.

In [ ]:
import os, glob, shutil, subprocess
REPO = ""   # e.g. "https://github.com/<user>/space-invaders-rainbow.git" (optional)
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
PROJ = f"{WORK}/RL"
if not os.path.exists(f"{PROJ}/train.py"):
    if REPO:
        subprocess.run(["git", "clone", "-q", REPO, PROJ], check=True)
    else:
        src = [p for p in glob.glob("/kaggle/input/**/train.py", recursive=True)] + glob.glob(f"{WORK}/**/train.py", recursive=True)
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob(f"{WORK}/*.zip")
        if src:
            shutil.copytree(os.path.dirname(src[0]), PROJ, dirs_exist_ok=True)
        elif zips:
            shutil.unpack_archive(zips[0], WORK)
            found = glob.glob(f"{WORK}/**/train.py", recursive=True)[0]
            if os.path.dirname(found) != PROJ:
                shutil.copytree(os.path.dirname(found), PROJ, dirs_exist_ok=True)
        else:
            raise SystemExit("Upload the RL folder (or RL.zip) first")
os.chdir(PROJ)
print(os.getcwd(), os.listdir())

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -r requirements.txt
!python tests/test_all.py

## 1. Speed test (~2 minutes)
Learning starts after 2 000 steps so the measured speed includes gradient updates.

In [ ]:
!python train.py --id speedtest --steps 200000 --learn-start 2000 --eval-interval 1000000 --log-interval 2000 \
    --checkpoint-interval 0 --time-limit 0.035 2>&1 | grep -E "steps/s|time limit" | tail -3

In [ ]:
STEPS_PER_SEC = 150   # <-- put the steps/s printed above
for steps in (500_000, 1_000_000, 2_000_000):
    print(f"{steps:>9,} steps ({steps * 4 / 1e6:.0f}M frames) -> {steps / STEPS_PER_SEC / 3600:5.1f} h")

## 2. Main run
`--time-limit 11` stops cleanly before Kaggle's 12 h limit and saves a checkpoint (+ replay buffer).
After the session ends: *Save Version* keeps `/kaggle/working`; in a new session add that output as
input, copy `runs/` back and run the same cell — `--resume` continues from the checkpoint.

In [ ]:
RUN_ID, STEPS = "rainbow_pixels", 1_000_000
# copy runs from a previous session's output, if attached
for prev in glob.glob(f"/kaggle/input/**/runs/{RUN_ID}", recursive=True):
    shutil.copytree(prev, f"runs/{RUN_ID}", dirs_exist_ok=True)
resume = "--resume" if os.path.exists(f"runs/{RUN_ID}/checkpoint.pt") else ""
!python train.py --id {RUN_ID} --steps {STEPS} --time-limit 11 --save-memory {resume}

In [ ]:
from IPython.display import Image, display
import pandas as pd
display(Image(f"runs/{RUN_ID}/learning_curve.png"))
display(pd.read_csv(f"runs/{RUN_ID}/eval.csv").tail(10))

## 3. Optional: baseline and ablations (one per session)
```
!python train.py --id dqn       --dqn          --steps 1000000 --time-limit 11
!python train.py --id no_per    --no-per       --steps 1000000 --time-limit 11
!python train.py --id shaped    --reward shaped --steps 1000000 --time-limit 11
!python train.py --id curriculum --curriculum "0:0,200000:1" --steps 1000000 --time-limit 11
!python train.py --id vector    --obs vector   --steps 1000000 --time-limit 11
```

## 4. Evaluation vs random and demo video

In [ ]:
!python evaluate.py --models runs/{RUN_ID}/model.pth --names Rainbow --episodes 30
!python demo.py --model runs/{RUN_ID}/model.pth --compare-random --out demo_vs_random.mp4
from IPython.display import Video
Video("demo_vs_random.mp4", embed=True, width=720)

In [ ]:
# pack everything to download (the replay buffer is left out - it is large)
shutil.copytree(f"runs/{RUN_ID}", f"/tmp/pack/runs/{RUN_ID}", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("memory.npz"))
shutil.make_archive(f"{WORK}/results_{RUN_ID}", "zip", "/tmp/pack")
for f in ("results", "demo_vs_random.mp4"):
    if os.path.exists(f):
        shutil.copy2(f, WORK) if os.path.isfile(f) else shutil.copytree(f, f"{WORK}/{f}", dirs_exist_ok=True)
print(os.listdir(WORK))